# Jalur B otomatis, melatih model dari nol

Notebook ini menjalankan seluruh Jalur B dalam **satu sel**. Klon kode, pasang
dependensi, siapkan dataset, verifikasi datanya, pasang penyesuai presisi GPU,
melatih, lalu meringkas hasilnya.

## Dua hal yang harus Anda lakukan sendiri

Keduanya tidak dapat diotomatiskan karena Colab memang meminta interaksi
manusia untuk itu. Notebook akan berhenti dengan pesan jelas bila salah satunya
terlewat, bukan gagal di tengah jalan.

**Satu, atur GPU sebelum menjalankan apa pun.** Menu **Runtime → Change runtime
type → T4 GPU → Save**. Mengganti tipe runtime akan me-restart dan menghapus
semua isinya, jadi lakukan ini lebih dulu, bukan sesudah sel berjalan.

**Dua, izinkan akses Google Drive.** Di tengah proses akan muncul jendela
permintaan izin. Klik **Connect to Google Drive** lalu pilih akun dan setujui.
Drive dipakai untuk mengambil arsip dataset yang sudah ada di sana, sehingga
tidak perlu mengunduh satu gigabyte lagi.

Di luar itu tidak ada yang perlu dipasang, tidak di komputer Anda maupun di
tempat lain. Semua berjalan di dalam Colab.

## Perkiraan waktu

| Tahap | Perkiraan |
|---|---|
| Klon kode dan pasang dependensi | 1 menit |
| Menyiapkan dataset dari Drive | 3 sampai 5 menit |
| Memverifikasi 17.870 berkas | 2 menit |
| Melatih `cnn_asp` 10 epoch | 15 sampai 25 menit |

Untuk mencoba lebih dulu apakah semuanya jalan, setel **EPOCHS ke 1**. Selesai
dalam beberapa menit, dan bila lancar Anda boleh menaikkannya dengan tenang.


In [ ]:
#@title ▶ Jalankan seluruh Jalur B { display-mode: "form" }
#@markdown Ubah pilihan di bawah bila perlu, lalu klik tombol play di sebelah kiri.
#@markdown Seluruh tahapan berjalan otomatis sesudahnya.

MODEL = "cnn_asp" #@param ["cnn_asp", "cnnlstm", "wav2vec2", "ast", "wavlm", "hubert", "nes2net"]
AUGMENTASI = "codec" #@param ["codec", "full", "fullbg", "fullrb", "fullbgrb", "none", "proposal"]
EPOCHS = 10 #@param {type:"slider", min:1, max:20, step:1}
BATCH = 32 #@param {type:"slider", min:8, max:64, step:8}
SEED = 42 #@param {type:"integer"}
SIMPAN_HASIL_KE_DRIVE = False #@param {type:"boolean"}

# =====================================================================
import os, shutil, subprocess, sys, time

AKAR = "/content/general-ai"
REPO = "https://github.com/Tristan-tech-ai/general-AI.git"
MULAI = time.time()

def fase(n, judul):
    print(f"\n{'=' * 68}\n  FASE {n}  {judul}\n{'=' * 68}")

def berhenti(pesan):
    print("\n" + "!" * 68)
    print("  BERHENTI")
    print("!" * 68)
    print(pesan)
    raise SystemExit(1)

def jalankan(perintah, cwd=None, tampilkan=True):
    # Keluaran dialirkan baris demi baris supaya kemajuan pelatihan terlihat,
    # bukan menghilang selama dua puluh menit lalu muncul sekaligus.
    p = subprocess.Popen(perintah, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True,
                         encoding="utf-8", errors="replace", bufsize=1)
    baris_akhir = ""
    for baris in p.stdout:
        if tampilkan:
            print(baris, end="")
        if baris.strip():
            baris_akhir = baris.strip()
    p.wait()
    return p.returncode, baris_akhir

# ---------------------------------------------------------------- 0
fase(0, "Memeriksa runtime")
try:
    import torch
except ImportError:
    berhenti("PyTorch tidak ada. Runtime Colab tampaknya tidak standar.")

if not torch.cuda.is_available():
    berhenti(
        "GPU tidak aktif, sedangkan Jalur B memerlukannya.\n\n"
        "Perbaikannya:\n"
        "  1. Menu Runtime -> Change runtime type\n"
        "  2. Pilih T4 GPU, lalu Save\n"
        "  3. Runtime akan restart dan seluruh isinya terhapus\n"
        "  4. Jalankan sel ini lagi dari awal\n\n"
        "Lakukan langkah itu lebih dulu, sebelum menjalankan sel apa pun.")

cc = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU      : {torch.cuda.get_device_name(0)}")
print(f"kemampuan: compute capability {cc[0]}.{cc[1]}, {vram:.1f} GB")
print(f"presisi  : {'bfloat16 asli' if cc[0] >= 8 else 'float16, bf16 hanya emulasi di kartu ini'}")
print(f"CPU      : {os.cpu_count()} core")
_, _, bebas = shutil.disk_usage("/content")
print(f"disk     : {bebas / 1e9:.0f} GB bebas")
if bebas < 8e9:
    berhenti("Ruang disk kurang dari 8 GB. Dataset perlu sekitar 3 GB.")

# ---------------------------------------------------------------- 1
fase(1, "Mengambil kode")
if os.path.exists(os.path.join(AKAR, ".git")):
    print("sudah ada, memperbarui ...")
    jalankan(["git", "pull", "--quiet"], cwd=AKAR)
else:
    shutil.rmtree(AKAR, ignore_errors=True)
    kode_keluar, _ = jalankan(["git", "clone", "--quiet", REPO, AKAR])
    if kode_keluar != 0:
        berhenti("Klon gagal. Periksa koneksi internet Colab.")
os.chdir(AKAR)
sys.path.insert(0, AKAR)
jalankan(["git", "log", "--oneline", "-1"])

# ---------------------------------------------------------------- 2
fase(2, "Memasang dependensi")
# Hanya yang benar-benar belum ada. Colab sudah membawa torch, torchaudio,
# numpy, scipy, dan sklearn.
perlu = []
for modul, paket in [("soundfile", "soundfile"), ("reportlab", "reportlab")]:
    try:
        __import__(modul)
    except ImportError:
        perlu.append(paket)
# transformers hanya diperlukan oleh model berbasis SSL dan AST
if MODEL not in ("cnn_asp", "cnnlstm"):
    try:
        import transformers
        print(f"transformers {transformers.__version__} sudah ada")
    except ImportError:
        perlu.append("transformers")
if perlu:
    print("memasang:", ", ".join(perlu))
    jalankan([sys.executable, "-m", "pip", "install", "--quiet", *perlu],
             tampilkan=False)
print("dependensi siap")

# ---------------------------------------------------------------- 3
fase(3, "Pengaman konfigurasi")
kode_keluar, _ = jalankan([sys.executable, "cek_konfigurasi.py"])
if kode_keluar != 0:
    berhenti("Pengaman konfigurasi menolak melanjutkan. Baca keluarannya di atas.")

# ---------------------------------------------------------------- 4
fase(4, "Menyiapkan dataset")
import hashlib, json, zipfile

ACUAN = json.load(open("dataset_acuan.json", encoding="utf-8"))
SHA_HARAP, BYTES_HARAP = ACUAN["arsip"]["sha256"], ACUAN["arsip"]["bytes"]
TUJUAN = "data/for-2seconds"

def sha256(p, blok=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(blok), b""):
            h.update(b)
    return h.hexdigest()

def cari_akar(mulai):
    for dp, _, _ in os.walk(mulai):
        if (os.path.isdir(os.path.join(dp, "training", "real"))
                and os.path.isdir(os.path.join(dp, "testing", "fake"))):
            return dp
    return None

if os.path.isdir(os.path.join(TUJUAN, "training", "real")):
    print("dataset sudah terpasang di runtime ini, penyiapan dilewati")
else:
    print("Meminta izin akses Google Drive.")
    print("Bila muncul jendela permintaan izin, klik Connect to Google Drive.\n")
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"

    KANDIDAT = [
        ("arsip penelitian yang sudah ada di Drive ini",
         f"{DRIVE}/PenelitianAudioDeepfake/Dataset/FoR.zip", "zip"),
        ("simpanan notebook ini",
         f"{DRIVE}/dataset-for/for-2sec.tar.gz", "targz"),
    ]
    ARSIP = JENIS = ASAL = None
    for nama, path, jenis in KANDIDAT:
        if os.path.exists(path):
            ARSIP, JENIS, ASAL = path, jenis, nama
            print(f"memakai {nama}")
            print(f"  {path}")
            print(f"  {os.path.getsize(path) / 1e9:.2f} GB, tidak ada yang diunduh")
            break
    if ARSIP is None:
        print("tidak ada arsip di Drive, mengunduh dari York University ...")
        os.makedirs(f"{DRIVE}/dataset-for", exist_ok=True)
        ARSIP = f"{DRIVE}/dataset-for/for-2sec.tar.gz"
        JENIS, ASAL = "targz", "unduhan baru"
        kode_keluar, _ = jalankan(["wget", "-c", "--no-verbose", "--show-progress",
                                   "-O", ARSIP, ACUAN["sumber"]])
        if kode_keluar != 0:
            berhenti("Unduhan gagal. Coba jalankan sel ini lagi, wget melanjutkan "
                     "dari bagian yang sudah terunduh.")

    if JENIS == "targz":
        print("\nmemeriksa sha256 arsip ...")
        besar, sha = os.path.getsize(ARSIP), sha256(ARSIP)
        print(f"  ukuran : {besar}  (acuan {BYTES_HARAP})")
        print(f"  sha256 : {sha}")
        if besar != BYTES_HARAP or sha != SHA_HARAP:
            berhenti("Arsip berbeda dari acuan. Hapus berkas itu lalu ulangi. "
                     "Bila tetap berbeda, penerbitnya mengganti arsip dan angka "
                     "dari sini tidak sebanding dengan angka di repositori.")
        print("  arsip sama dengan yang dipakai di mesin lokal")
    else:
        print("\nsha256 arsip dilewati, wadahnya zip sedangkan acuan dibuat dari")
        print("tar.gz. Kesamaan isinya diperiksa lewat sidik pohon di fase 5.")

    print("\nmengekstrak ...")
    TMP = "data/_arsip"
    os.makedirs("data", exist_ok=True)
    shutil.rmtree(TMP, ignore_errors=True)
    if JENIS == "targz":
        jalankan(["tar", "-xzf", ARSIP, "-C", "data"], tampilkan=False)
        sumber = cari_akar("data")
    else:
        with zipfile.ZipFile(ARSIP) as z:
            z.extractall(TMP)
        sumber = cari_akar(TMP)
    if not sumber:
        berhenti(f"Struktur training/validation/testing tidak ditemukan di dalam "
                 f"arsip. Isi data/: {os.listdir('data')}")
    if os.path.abspath(sumber) != os.path.abspath(TUJUAN):
        shutil.rmtree(TUJUAN, ignore_errors=True)
        shutil.move(sumber, TUJUAN)
    shutil.rmtree(TMP, ignore_errors=True)
    print(f"dataset siap, sumbernya {ASAL}")

total = 0
for s in ["training", "validation", "testing"]:
    for c in ["real", "fake"]:
        n = len(os.listdir(f"{TUJUAN}/{s}/{c}"))
        total += n
        print(f"  {s:<11} {c:<5} {n:>6} berkas")
print(f"  {'TOTAL':<17} {total:>6} berkas   (seharusnya 17.870)")
if total != 17870:
    berhenti(f"Jumlah berkas {total}, seharusnya 17.870. Datasetnya tidak utuh.")

# ---------------------------------------------------------------- 5
fase(5, "Memverifikasi dataset")
kode_keluar, akhir = jalankan([sys.executable, "cek_dataset.py"])
if kode_keluar != 0:
    berhenti("Dataset BERBEDA dari acuan.\n\n"
             "Jangan melanjutkan. Angka yang dihasilkan dari dataset yang "
             "berbeda tidak boleh dibandingkan dengan angka di repositori.")

# ---------------------------------------------------------------- 6
fase(6, "Menyiapkan pelatihan")
if os.path.exists("manifest.csv"):
    os.remove("manifest.csv")
from forlib.data import build_manifest
baris = build_manifest(TUJUAN, "manifest.csv")
print(f"manifest dibangun ulang: {len(baris)} baris")
print(f"contoh path: {baris[0]['path']}")

kode_keluar, _ = jalankan([sys.executable, "colab/colab_patch.py", "--apply"])
if kode_keluar != 0:
    berhenti("Patch presisi gagal dipasang. Baca keluarannya di atas.")

# ---------------------------------------------------------------- 7
fase(7, f"Melatih {MODEL}")
print(f"model      : {MODEL}")
print(f"augmentasi : {AUGMENTASI}")
print(f"epoch      : {EPOCHS}   batch: {BATCH}   seed: {SEED}")
print(f"keluaran   : runs_colab/  (sengaja terpisah dari runs/, jangan digabung)")
print(f"worker     : 2  (Colab gratis hanya punya dua core)\n")
t0 = time.time()
kode_keluar, _ = jalankan([
    sys.executable, "train.py",
    "--model", MODEL, "--split", "official", "--augment", AUGMENTASI,
    "--epochs", str(EPOCHS), "--batch", str(BATCH), "--workers", "2",
    "--seed", str(SEED), "--out", "runs_colab",
])
if kode_keluar != 0:
    berhenti("Pelatihan gagal. Kalau pesannya menyebut kehabisan memori GPU, "
             "turunkan BATCH menjadi 16 lalu jalankan sel ini lagi.")
print(f"\npelatihan selesai dalam {(time.time() - t0) / 60:.1f} menit")

# ---------------------------------------------------------------- 8
fase(8, "Ringkasan hasil")
import glob
import numpy as np
from forlib.metrics import full_metrics, prior_matched_threshold

print(f"{'run':<44}{'acc@0,5':>9}{'acc@prior':>11}{'AUC':>9}{'EER':>9}   asal")
print("-" * 86)
for d in sorted(glob.glob("runs_colab/*")) + sorted(glob.glob("runs/*_official_*_s42")):
    f = os.path.join(d, "test_scores.npy")
    if not os.path.exists(f):
        continue
    if not d.startswith("runs_colab") and MODEL not in os.path.basename(d):
        continue
    y, p, _ = np.load(f)
    y = y.astype(int)
    m0 = full_metrics(y, p, 0.5)
    mp = full_metrics(y, p, prior_matched_threshold(p, 0.5))
    asal = "Colab" if d.startswith("runs_colab") else "lokal"
    print(f"{os.path.basename(d):<44}{m0['accuracy']*100:>8.2f}%"
          f"{mp['accuracy']*100:>10.2f}%{m0['auc']:>9.4f}{m0['eer']*100:>8.2f}%   {asal}")
print("-" * 86)
print("Ragam antar inisialisasi pada konfigurasi semacam ini sekitar 3,5 poin")
print("persentase atas tiga seed. Selisih yang lebih kecil dari itu belum dapat")
print("ditafsirkan sebagai perbedaan.")

# ---------------------------------------------------------------- 9
if SIMPAN_HASIL_KE_DRIVE:
    fase(9, "Menyimpan hasil ke Drive")
    from google.colab import drive
    drive.mount("/content/drive")
    TUJUAN_DRIVE = "/content/drive/MyDrive/general-ai-hasil-colab"
    n = 0
    for pola in ["runs_colab/*/results.json", "runs_colab/*/test_scores.npy"]:
        for src in glob.glob(pola):
            dst = os.path.join(TUJUAN_DRIVE, src)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
            n += 1
    print(f"{n} berkas tersimpan di {TUJUAN_DRIVE}")
    print("Bobot model tidak ikut disalin karena ukurannya bisa lebih dari satu")
    print("gigabyte, sedangkan seluruh analisis hanya memerlukan skornya.")
else:
    print("\nHasil hanya ada di runtime ini dan akan hilang ketika sesi berakhir.")
    print("Setel SIMPAN_HASIL_KE_DRIVE menjadi True bila ingin menyimpannya.")

print(f"\n{'=' * 68}")
print(f"  SELESAI dalam {(time.time() - MULAI) / 60:.1f} menit")
print(f"{'=' * 68}")


---

## Kalau berhenti di tengah jalan

Notebook ini berhenti dengan pesan yang menjelaskan sebabnya, bukan dengan
tumpukan galat. Tiga yang paling sering muncul:

**GPU tidak aktif.** Atur Runtime → Change runtime type → T4 GPU, lalu jalankan
sel dari awal.

**Kehabisan memori GPU.** Turunkan `BATCH` menjadi 16, atau 8 untuk model
besar seperti `wavlm` dan `hubert`.

**Dataset berbeda dari acuan.** Ini bukan kesalahan Anda dan justru pengaman
yang bekerja. Artinya salinan dataset di Drive itu tidak identik dengan yang
dipakai di repositori, sehingga angkanya tidak sebanding. Sebabnya harus
diketahui lebih dulu sebelum dilanjutkan.

## Menjalankan ulang

Sel ini aman dijalankan berkali-kali. Kode diperbarui dengan `git pull`,
dataset yang sudah terpasang tidak disiapkan ulang, dan patch presisi tidak
terpasang dua kali. Yang berubah hanya hasil pelatihannya.

Untuk mencoba model lain, ubah pilihan `MODEL` lalu jalankan lagi. Hasil
sebelumnya tetap tersimpan di `runs_colab/` karena nama direktorinya memuat
model, augmentasi, batch, epoch, dan seed.

## Satu aturan yang tidak boleh dilanggar

Hasil dari sini masuk ke `runs_colab/`, bukan `runs/`. Jangan pernah
menggabungkan keduanya. Perangkat keras dan presisinya berbeda, sehingga
penggabungan itu akan melaporkan ragam antar perangkat sebagai ragam antar
inisialisasi acak. Itu persis kelas kekeliruan yang tujuh kali terjadi dalam
penelitian ini, dan `cek_konfigurasi.py` tidak dapat menangkapnya karena ia
hanya memeriksa jumlah epoch dan ukuran batch.
